In [ ]:
import numpy as np
import math

In [ ]:
import pandas as pd
import glob
import os

# ---- Параметры -------------------------------------------------------------
FOLDER_WITH_EXCEL = r"./LP_SP"    # папка, где лежат .xlsx
PATTERN = "*.xlsx"                      # при необходимости можно изменить на *.xlsm и т.д.
ADD_SOURCE_COLUMNS = True               # добавить колонки с именем файла/листа
# ---------------------------------------------------------------------------

def read_all_excels(folder, pattern="*.xlsx", add_source_cols=True):
    """
    Читать все Excel-файлы из папки и вернуть один объединённый DataFrame.

    :param folder: Папка с .xlsx
    :param pattern: Глоб-шаблон для поиска файлов
    :param add_source_cols: Добавлять ли столбцы __file__ и __sheet__
    """
    full_pattern = os.path.join(folder, pattern)
    files = glob.glob(full_pattern)

    if not files:
        raise FileNotFoundError(f"По шаблону {full_pattern!r} ничего не найдено")

    frames = []  # сюда будем складывать все DataFrame'ы

    for fname in files:
        # Читаем сразу все листы: вернётся словарь {имя_листа: DataFrame}
        sheets_dict = pd.read_excel(fname, sheet_name=None, engine="openpyxl")

        for sheet_name, df in sheets_dict.items():
            if add_source_cols:
                df = df.copy()  # чтобы не трогать оригинал
                df["file"]  = os.path.basename(fname)
                df["sheet"] = sheet_name
            frames.append(df)

    # Объединяем
    combined = pd.concat(frames, ignore_index=True)
    return combined

In [ ]:
df_all = read_all_excels(FOLDER_WITH_EXCEL)

In [ ]:
df_all['mu'] = [float(x.replace('.xlsx','').replace('mu=','')) for x in df_all['file']]

In [ ]:
df_all.head()

In [ ]:
df_LP = df_all[df_all['sheet'] == 'LP']

In [ ]:
import scipy.io as sio
import numpy as np
import matplotlib.pyplot as plt
import pysr 
from pysr import PySRRegressor
from scipy.optimize import least_squares
from sympy.utilities.codegen import codegen
from scipy.integrate import solve_ivp

In [ ]:
def odeCR3BP(t, p, mu):
    R = np.sqrt((p[0] + mu)**2 + p[1]**2)
    r = np.sqrt((p[0] - 1 + mu)**2 + p[1]**2)
    dpdt = np.array([p[2], p[3],
    2 * p[3] + p[0] - (1 - mu) * (p[0] + mu) / R**3 - mu * (p[0] - 1 + mu) / r**3,
    -2 * p[2] + p[1] - (1 - mu) * p[1] / R**3 - mu * p[1] / r**3])

    return dpdt

In [ ]:
fig = plt.figure(1)
ax = fig.add_subplot(111, projection='3d')
plt.gca().set_facecolor('white')
ax.scatter(df_LP['a'],df_LP['mu'],df_LP['v1'])
ax.set_xlabel('a')
ax.set_ylabel('mu')
ax.set_zlabel('v1')

In [ ]:
mu_list = list(set(df_LP['mu']))
for mu in mu_list:
    df_LP_mu = df_LP[df_LP['mu'] == mu]
    plt.plot(df_LP_mu['a'],df_LP_mu['v1'])

In [ ]:
fig = plt.figure(1)
ax = fig.add_subplot(111, projection='3d')
plt.gca().set_facecolor('white')
ax.scatter(df_LP['a'],df_LP['mu'],df_LP['v2'])
ax.set_xlabel('a')
ax.set_ylabel('mu')
ax.set_zlabel('x1')

In [ ]:
mu_list

In [ ]:
mu_list = list(set(df_LP['mu']))
for mu in mu_list:
    df_LP_mu = df_LP[df_LP['mu'] == mu]
    plt.plot(df_LP_mu['a'],df_LP_mu['v2'])

In [ ]:
fig = plt.figure(1)
ax = fig.add_subplot(111, projection='3d')
plt.gca().set_facecolor('white')
ax.scatter(df_LP['a'],df_LP['mu'],df_LP['T'])

In [ ]:
mu = mu_list[1]

df_SP_mu = df_SP[df_SP['mu'] == mu]

a = np.array(df_SP_mu['a'])
x1 = np.array(df_SP_mu['x1'])
x2 = np.array(df_SP_mu['x2'])
v1 = np.array(df_SP_mu['v1'])
v2 = np.array(df_SP_mu['v2'])
T = np.array(df_SP_mu['T'])

In [ ]:
fig_man, ax_man = plt.subplots(1)
fig_res, ax_res = plt.subplots(1)
res_list = []
for i in range(0, x1.size, 5):
    #print(i)
    p0 = np.array([x1[i], x2[i], v1[i], v2[i]])
    p = solve_ivp(lambda t, p: odeCR3BP(t, p, mu),
                  t_span=(0, T[i]),
                  y0=p0,
                  method='LSODA',
                  t_eval = np.linspace(0, T[i], 1000),
                  atol=1e-12,
                  rtol=1e-12)
    ax_man.plot(p.y[0,:], p.y[1,:])
    res = np.linalg.norm(p.y[:,-1] - p0)
    ax_res.plot(x1[i], res, '.k')
    res_list.append(res)
ax_man.axis('equal')
plt.grid()
plt.show()

In [ ]:
# %%
L1 =  0.836915131427382
L2 =  1.155682161024677
L3 = -1.005062645331442

In [ ]:
plt.plot(a, v1)
plt.grid()
plt.xlabel('a')
plt.ylabel('v1')
#plt.legend()
plt.show()

In [ ]:
plt.plot(a, v2)
plt.grid()
plt.xlabel('a')
plt.ylabel('v2')
#plt.legend()
plt.show()

In [ ]:
plt.plot(a, T)
plt.grid()
plt.xlabel('a')
plt.ylabel('T')
#plt.legend()
plt.show()

In [ ]:
def x1_fun(a, mu):
    #y = (-5.3891767e-8*x - math.sin(x - 0.52359873) - 0.029999906) #3.7e-08
    y = (-mu + math.sin(a + np.pi*5/6))
    return y
def x2_fun(a, mu):
    #y = (math.sin(x - 2.0943952)) #6.1e-08
    y = (math.sin(a - np.pi*2/3))
    return y
def v1_fun(x):
    y = (-0.31065616*((x + 0.04729554*math.sin(1.2135408*x + 1.6476246))*(5.527562*x + math.cos(1.7415938*x**1.5302258)) - 0.04729554)*math.sin(0.8843217*x + 2.52133)/(5.527562*x + math.cos(1.7415938*x**1.5302258))) #1.4e-04
    return y
def v2_fun(x):
    y = ((x - 0.0032811477*(0.83023274*x)**x)*(x*(-0.076037794*x + 0.15013573*math.sin(x) + 0.11507184*math.cos(x) + 0.20255086) + 0.002156449)/x) #1.1e-04
    return y
def T_fun(x):
    y = ((1.3619989 - 0.056624785*0.061825007**x)*(0.48113042**x + 0.05254563*(x + math.cos(0.37977582*x**2))**0.07269894*math.cos(x - 0.62925476) + 4.5865707)) #2.6e-04
    return y
    
x_range = np.linspace(np.min(a), np.max(a), 100)

fig_man, ax_man = plt.subplots(1)
fig_res, ax_res = plt.subplots(1)
for i in range(x_range.size):
    #print(i)
    p0 = np.array([x1_fun(x_range[i], mu), x2_fun(x_range[i], mu), v1_fun(x_range[i]), v2_fun(x_range[i])])
    p = solve_ivp(lambda t, p: odeCR3BP(t, p, mu),
                  t_span=(0, T_fun(x_range[i])),
                  y0=p0,
                  method='LSODA',
                  t_eval = np.linspace(0, T_fun(x_range[i]), 1000),
                  atol=1e-12,
                  rtol=1e-12)
    ax_man.plot(p.y[0,:], p.y[1,:])
    ax_res.plot(x_range[i], np.linalg.norm(p.y[:,-1] - p0), '.k')
ax_man.axis('equal')
plt.grid()
plt.show()


In [ ]:
2.617994/np.pi

In [ ]:
np.pi*5/6

In [ ]:
plt.plot(a, v1)
plt.plot(a, [v1_fun(x) for x in a],'--')
plt.grid()
plt.xlabel('a')
plt.ylabel('v1')
#plt.legend()
plt.show()

In [ ]:
plt.plot(a, v2)
plt.plot(a, [v2_fun(x) for x in a],'--')
plt.grid()
plt.xlabel('a')
plt.ylabel('v1')
#plt.legend()
plt.show()

In [ ]:
plt.plot(a, x1)
plt.plot(a, [x1_fun(x, mu) for x in a],'--')
plt.grid()
plt.xlabel('a')
plt.ylabel('v1')
#plt.legend()
plt.show()

In [ ]:
plt.plot(a, x2)
plt.plot(a, [x2_fun(x) for x in a],'--')
plt.grid()
plt.xlabel('a')
plt.ylabel('v1')
#plt.legend()
plt.show()

In [ ]:
2.0943952/np.pi

In [ ]:
2/3*np.pi